In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

df = pd.read_csv('data/raw/PS_20174392719_1491204439457_log.csv')

# Build a profile for every unique sender customer
customer_profile = df.groupby('nameOrig').agg(
    txn_count=('amount', 'count'),
    total_sent=('amount', 'sum'),
    avg_txn=('amount', 'mean'),
    max_txn=('amount', 'max'),
    unique_types=('type', 'nunique'),
    active_days=('step', 'nunique')
).reset_index()

print(f"Unique customers: {len(customer_profile):,}")
print(f"\nCustomer profile sample:")
print(customer_profile.head())
print(f"\nTransaction count stats:")
print(customer_profile['txn_count'].describe())

Unique customers: 6,353,307

Customer profile sample:
      nameOrig  txn_count  total_sent    avg_txn    max_txn  unique_types  \
0  C1000000639          1   244486.46  244486.46  244486.46             1   
1  C1000001337          1     3170.28    3170.28    3170.28             1   
2  C1000001725          1     8424.74    8424.74    8424.74             1   
3  C1000002591          1   261877.19  261877.19  261877.19             1   
4  C1000003372          1    20528.65   20528.65   20528.65             1   

   active_days  
0            1  
1            1  
2            1  
3            1  
4            1  

Transaction count stats:
count    6.353307e+06
mean     1.001466e+00
std      3.832002e-02
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.000000e+00
Name: txn_count, dtype: float64


In [ ]:
# Segment based on transaction activity
def segment(row):
    if row['txn_count'] >= 10:
        return 'Active'
    elif row['txn_count'] >= 3:
        return 'Moderate'
    else:
        return 'Thin-file'

customer_profile['segment'] = customer_profile.apply(segment, axis=1)

# Summary per segment
summary = customer_profile.groupby('segment').agg(
    customer_count=('nameOrig', 'count'),
    avg_transactions=('txn_count', 'mean'),
    avg_total_sent=('total_sent', 'mean'),
    total_value=('total_sent', 'sum')
).reset_index()

summary['pct_customers'] = summary['customer_count'] / summary['customer_count'].sum() * 100
summary['pct_value'] = summary['total_value'] / summary['total_value'].sum() * 100

print("=== CUSTOMER SEGMENTS ===")
print(summary.to_string(index=False))